# Load Keystone Eval Results from S3

Loads every `eval_result.json` under an S3 prefix, validates each as a
`KeystoneRepoResult`, and flattens into a single DataFrame.

---

> **Note:** This notebook is checked in without cell outputs.  
> To execute and save results to a separate file, run:
> ```bash
> uv run jupyter nbconvert --to notebook --execute evals/eda/load_s3_eval_results.ipynb --output load_s3_eval_results_executed.ipynb
> ```

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on sys.path so 'evals' and 'keystone' are importable
# regardless of which Jupyter kernel is active.
_project_root = str(Path.cwd().resolve().parents[1])  # evals/eda -> project root
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

import json

import pandas as pd
import s3fs
from pydantic import ValidationError

from eval_schema import KeystoneRepoResult

import plotly.express as px

In [ ]:
S3_PREFIX = "s3://int8-datasets/keystone/evals/2026-03-08_four_model_thad_v2"
PARQUET_CACHE = Path("2026-03-08_four_model_thad_v2.parquet")

In [ ]:
def _build_record(r: KeystoneRepoResult) -> dict:
    """Flatten a KeystoneRepoResult into a dict for the DataFrame."""
    br = r.bootstrap_result
    agent = br.agent if br else None
    ver = br.verification if br else None
    cost = agent.cost if agent else None

    config_name = None
    if r.keystone_config and r.keystone_config.agent_config:
        model = r.keystone_config.agent_config.model
        config_name = model.value if model else None

    return {
        # Full result as JSON string (first column)
        "raw_json": r.model_dump_json(),
        "repo_id": r.repo_entry.id,
        "config_name": config_name,
        "trial_index": r.trial_index,
        "success": r.success,
        "error_message": r.error_message,
        "agent_exit_code": agent.exit_code if agent else None,
        "agent_walltime_seconds": agent.duration_seconds if agent else None,
        "agent_timed_out": agent.timed_out if agent else None,
        "keystone_exit_code": 0 if r.success else 1,
        "cost_usd": cost.cost_usd if cost else None,
        "input_tokens": cost.token_spending.input if cost else None,
        "output_tokens": cost.token_spending.output if cost else None,
        "image_build_seconds": ver.image_build_seconds if ver else None,
        "test_execution_seconds": ver.test_execution_seconds if ver else None,
        "tests_passed": ver.tests_passed if ver else None,
        "tests_failed": ver.tests_failed if ver else None,
        "summary": agent.summary.message if agent and agent.summary else None,
        "status_messages": json.dumps([
            {"timestamp": sm.timestamp, "message": sm.message}
            for sm in (agent.status_messages if agent else [])
        ]),
    }


def _load_from_s3() -> pd.DataFrame:
    """Fetch every eval_result.json from S3, validate, and build a DataFrame."""
    fs = s3fs.S3FileSystem(anon=False)
    prefix_path = S3_PREFIX.removeprefix("s3://")
    json_paths = fs.glob(f"{prefix_path}/**/eval_result.json")
    print(f"Found {len(json_paths)} eval_result.json files in S3")

    records = []
    errors = []
    for s3_path in json_paths:
        with fs.open(s3_path, "r") as f:
            raw = json.load(f)
        try:
            result = KeystoneRepoResult(**raw)
            records.append(_build_record(result))
        except ValidationError as e:
            errors.append({"path": s3_path, "error": str(e)})

    print(f"Validated {len(records)} results, {len(errors)} validation errors")
    for err in errors[:5]:
        print(f"  FAIL: {err['path']}\n        {err['error'][:200]}")

    return pd.DataFrame(records)

In [ ]:
if PARQUET_CACHE.exists():
    df = pd.read_parquet(PARQUET_CACHE)
    print(f"Loaded {len(df)} rows from cache: {PARQUET_CACHE}")
else:
    df = _load_from_s3()
    df.to_parquet(PARQUET_CACHE, index=False)
    print(f"Saved {len(df)} rows to cache: {PARQUET_CACHE}")

print(f"DataFrame: {len(df)} rows, {len(df.columns)} columns")
df.head()


In [ ]:
px.ecdf(df, x="agent_walltime_seconds", color="config_name")

In [ ]:
df.info()

In [ ]:
# Quick summary by config
if "config_name" in df.columns and df["config_name"].notna().any():
    summary = (
        df.groupby("config_name")
        .agg(
            total=("success", "count"),
            successes=("success", "sum"),
            success_rate=("success", "mean"),
            mean_cost_usd=("cost_usd", "mean"),
            mean_agent_walltime=("agent_walltime_seconds", "mean"),
            mean_tests_passed=("tests_passed", "mean"),
            mean_tests_failed=("tests_failed", "mean"),
        )
    )
    summary["success_rate"] = (summary["success_rate"] * 100).round(1)
    summary